In [1]:
import os
# Find the latest version of spark 3.x  from http://www.apache.org/dist/spark/ and enter as the spark version
# For example:
# spark_version = 'spark-3.5.5'
spark_version = 'spark-3.5.5'
os.environ['SPARK_VERSION']=spark_version

# Install Spark and Java
!apt-get update
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
!wget -q http://www.apache.org/dist/spark/$SPARK_VERSION/$SPARK_VERSION-bin-hadoop3.tgz
!tar xf $SPARK_VERSION-bin-hadoop3.tgz
!pip install -q findspark

# Set Environment Variables
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["SPARK_HOME"] = f"/content/{spark_version}-bin-hadoop3"

# Start a SparkSession
import findspark
findspark.init()

Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1,381 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [8,812 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-sec

In [2]:
# Import packages
from pyspark.sql import SparkSession
import time

# Create a SparkSession
spark = SparkSession.builder.appName("SparkSQL").getOrCreate()

In [3]:
# 1. Read in the AWS S3 bucket into a DataFrame.
from pyspark import SparkFiles
url = "https://2u-data-curriculum-team.s3.amazonaws.com/dataviz-classroom/v1.2/22-big-data/home_sales_revised.csv"

spark.sparkContext.addFile(url)
df = spark.read.csv(SparkFiles.get("home_sales_revised.csv"), sep=",", header=True, inferSchema=True)
df.show(truncate=False)


+------------------------------------+----------+----------+------+--------+---------+-----------+--------+------+----------+----+
|id                                  |date      |date_built|price |bedrooms|bathrooms|sqft_living|sqft_lot|floors|waterfront|view|
+------------------------------------+----------+----------+------+--------+---------+-----------+--------+------+----------+----+
|f8a53099-ba1c-47d6-9c31-7398aa8f6089|2022-04-08|2016      |936923|4       |3        |3167       |11733   |2     |1         |76  |
|7530a2d8-1ae3-4517-9f4a-befe060c4353|2021-06-13|2013      |379628|2       |2        |2235       |14384   |1     |0         |23  |
|43de979c-0bf0-4c9f-85ef-96dc27b258d5|2019-04-12|2014      |417866|2       |2        |2127       |10575   |2     |0         |0   |
|b672c137-b88c-48bf-9f18-d0a4ac62fb8b|2019-10-16|2016      |239895|2       |2        |1631       |11149   |2     |0         |0   |
|e0726d4d-d595-4074-8283-4139a54d0d63|2022-01-08|2017      |424418|3       |2      

In [4]:
# 2. Create a temporary view of the DataFrame.
# create a view from the DataFrame named 'home_sales' which allows SQL statements to query the data
df.createOrReplaceTempView('home_sales') # makes a 'table' named 'home_sales'


In [5]:
# 3. What is the average price for a four bedroom house sold per year, rounded to two decimal places?
query = """
SELECT YEAR(date) AS YEAR, ROUND(AVG(price), 2) AS AVERAGE_PRICE
FROM home_sales
WHERE bedrooms = 4
GROUP BY YEAR
ORDER BY YEAR DESC;
"""
spark.sql(query).show(truncate=False)


+----+-------------+
|YEAR|AVERAGE_PRICE|
+----+-------------+
|2022|296363.88    |
|2021|301819.44    |
|2020|298353.78    |
|2019|300263.7     |
+----+-------------+



In [6]:
# 4. What is the average price of a home for each year the home was built,
# that have 3 bedrooms and 3 bathrooms, rounded to two decimal places?
query = """
SELECT date_built AS YEAR_BUILT, ROUND(AVG(price), 2) AS AVERAGE_PRICE
FROM home_sales
WHERE bedrooms = 3 AND bathrooms = 3
GROUP BY YEAR_BUILT
ORDER BY YEAR_BUILT DESC;
"""
spark.sql(query).show(truncate=False)


+----------+-------------+
|YEAR_BUILT|AVERAGE_PRICE|
+----------+-------------+
|2017      |292676.79    |
|2016      |290555.07    |
|2015      |288770.3     |
|2014      |290852.27    |
|2013      |295962.27    |
|2012      |293683.19    |
|2011      |291117.47    |
|2010      |292859.62    |
+----------+-------------+



In [7]:
# 5. What is the average price of a home for each year the home was built,
# that have 3 bedrooms, 3 bathrooms, with two floors,
# and are greater than or equal to 2,000 square feet, rounded to two decimal places?
query = """
SELECT date_built AS YEAR_BUILT, ROUND(AVG(price), 2) AS AVERAGE_PRICE
FROM home_sales
WHERE bedrooms = 3 AND bathrooms = 3 AND floors = 2 AND sqft_living >= 2000
GROUP BY YEAR_BUILT
ORDER BY YEAR_BUILT DESC;
"""
spark.sql(query).show(truncate=False)


+----------+-------------+
|YEAR_BUILT|AVERAGE_PRICE|
+----------+-------------+
|2017      |280317.58    |
|2016      |293965.1     |
|2015      |297609.97    |
|2014      |298264.72    |
|2013      |303676.79    |
|2012      |307539.97    |
|2011      |276553.81    |
|2010      |285010.22    |
+----------+-------------+



In [8]:
# 6. What is the average price of a home per "view" rating, rounded to two decimal places,
# having an average home price greater than or equal to $350,000? Order by descending view rating.
# Although this is a small dataset, determine the run time for this query.

start_time = time.time()
query = """
SELECT view AS VIEW_RATING, ROUND(AVG(price), 2) AS AVERAGE_PRICE
FROM home_sales
GROUP BY VIEW_RATING
HAVING AVG(price) >= 350000
ORDER BY VIEW_RATING DESC;
"""
spark.sql(query).show(truncate=False)


print("--- %s seconds ---" % (time.time() - start_time))

+-----------+-------------+
|VIEW_RATING|AVERAGE_PRICE|
+-----------+-------------+
|100        |1026669.5    |
|99         |1061201.42   |
|98         |1053739.33   |
|97         |1129040.15   |
|96         |1017815.92   |
|95         |1054325.6    |
|94         |1033536.2    |
|93         |1026006.06   |
|92         |970402.55    |
|91         |1137372.73   |
|90         |1062654.16   |
|89         |1107839.15   |
|88         |1031719.35   |
|87         |1072285.2    |
|86         |1070444.25   |
|85         |1056336.74   |
|84         |1117233.13   |
|83         |1033965.93   |
|82         |1063498.0    |
|81         |1053472.79   |
+-----------+-------------+
only showing top 20 rows

--- 1.2040300369262695 seconds ---


In [9]:
# 7. Cache the the temporary table home_sales.
spark.sql("cache table home_sales")

DataFrame[]

In [10]:
# 8. Check if the table is cached.
spark.catalog.isCached('home_sales')

True

In [11]:
# 9. Using the cached data, run the last query above, that calculates
# the average price of a home per "view" rating, rounded to two decimal places,
# having an average home price greater than or equal to $350,000.
# Determine the runtime and compare it to the uncached runtime.

start_time = time.time()
Query = """
SELECT view AS VIEW_RATING, ROUND(AVG(price), 2) AS AVERAGE_PRICE
FROM home_sales
GROUP BY VIEW_RATING
HAVING AVG(price) >= 350000
ORDER BY VIEW_RATING DESC;
"""
spark.sql(query).show(truncate=False)


print("--- %s seconds ---" % (time.time() - start_time))


+-----------+-------------+
|VIEW_RATING|AVERAGE_PRICE|
+-----------+-------------+
|100        |1026669.5    |
|99         |1061201.42   |
|98         |1053739.33   |
|97         |1129040.15   |
|96         |1017815.92   |
|95         |1054325.6    |
|94         |1033536.2    |
|93         |1026006.06   |
|92         |970402.55    |
|91         |1137372.73   |
|90         |1062654.16   |
|89         |1107839.15   |
|88         |1031719.35   |
|87         |1072285.2    |
|86         |1070444.25   |
|85         |1056336.74   |
|84         |1117233.13   |
|83         |1033965.93   |
|82         |1063498.0    |
|81         |1053472.79   |
+-----------+-------------+
only showing top 20 rows

--- 0.5848751068115234 seconds ---


In [12]:
# 10. Partition by the "date_built" field on the formatted parquet home sales data
df.write.partitionBy("date_built").mode("overwrite").parquet("p_home_sales")

In [13]:
# 11. Read the parquet formatted data.
p_df = spark.read.parquet('p_home_sales')
p_df.show(truncate=False)

+------------------------------------+----------+------+--------+---------+-----------+--------+------+----------+----+----------+
|id                                  |date      |price |bedrooms|bathrooms|sqft_living|sqft_lot|floors|waterfront|view|date_built|
+------------------------------------+----------+------+--------+---------+-----------+--------+------+----------+----+----------+
|2ed8d509-7372-46d5-a9dd-9281a95467d4|2021-08-06|258710|3       |3        |1918       |9666    |1     |0         |25  |2015      |
|941bad30-eb49-4a78-b83a-87abb87a62db|2020-05-09|229896|3       |3        |2197       |8641    |1     |0         |3   |2015      |
|c797ca12-52cd-4b13-9338-183653619b11|2019-06-08|288650|2       |3        |2100       |10419   |2     |0         |7   |2015      |
|0cfe57f3-28c2-472c-9bc3-aaeac6807e62|2019-10-04|308313|3       |3        |1960       |9453    |2     |0         |2   |2015      |
|d715f295-2fbf-4e9a-a79b-b0437aed3777|2021-05-17|391574|3       |2        |1635    

In [14]:
# 12. Create a temporary table for the parquet data.
p_df.createOrReplaceTempView("par_home_sales")

In [15]:
# 13. Using the parquet DataFrame, run the last query above, that calculates
# the average price of a home per "view" rating, rounded to two decimal places,
# having an average home price greater than or equal to $350,000.
# Determine the runtime and compare it to the cached runtime.

start_time = time.time()
query = """
SELECT view AS VIEW_RATING, ROUND(AVG(price), 2) AS AVERAGE_PRICE
FROM par_home_sales
GROUP BY VIEW_RATING
HAVING AVG(price) >= 350000
ORDER BY VIEW_RATING DESC;
"""
spark.sql(query).show(truncate=False)

print("--- %s seconds ---" % (time.time() - start_time))

+-----------+-------------+
|VIEW_RATING|AVERAGE_PRICE|
+-----------+-------------+
|100        |1026669.5    |
|99         |1061201.42   |
|98         |1053739.33   |
|97         |1129040.15   |
|96         |1017815.92   |
|95         |1054325.6    |
|94         |1033536.2    |
|93         |1026006.06   |
|92         |970402.55    |
|91         |1137372.73   |
|90         |1062654.16   |
|89         |1107839.15   |
|88         |1031719.35   |
|87         |1072285.2    |
|86         |1070444.25   |
|85         |1056336.74   |
|84         |1117233.13   |
|83         |1033965.93   |
|82         |1063498.0    |
|81         |1053472.79   |
+-----------+-------------+
only showing top 20 rows

--- 0.9179270267486572 seconds ---


In [16]:
# 14. Uncache the home_sales temporary table.
spark.sql("uncache table home_sales")

DataFrame[]

In [17]:
# 15. Check if the home_sales is no longer cached

spark.catalog.isCached('home_sales')

False